# OP-FL v2 — 09. D'où vient la dispersion des prédictions sur les hôpitaux verrouillés ? (descriptif, sans label)

**Statut : exploratoire et descriptif.** Ce notebook ne modifie pas OP-Cal et ne cherche pas à récupérer C1b ou C2.

**Constat du 08.** Var(â) vaut 0,047 sur le développement, 0,057 sur le développement réduit à la taille des hôpitaux verrouillés, et 0,108 sur le test verrouillé.
L'hypothèse examinée est l'**extrapolation** : le méta-modèle prédirait pour des hôpitaux dont les descripteurs sont mal représentés par les 60 hôpitaux de développement.
C'est une hypothèse compatible avec les observations, pas un mécanisme démontré : une dispersion plus forte peut aussi refléter une hétérogénéité réelle.

**Données lues.**
* Hôpitaux de développement : descripteurs du 06b et cibles a_h, pour réajuster le méta-modèle final exactement comme au 07.
* Hôpitaux verrouillés : **uniquement** ce que la phase A du 07 a gelé avant l'ouverture, c'est-à-dire leurs descripteurs, les scores lp du modèle final (sans label) et les â_h.
* **Aucun label, aucune cible a_h, aucune performance** des hôpitaux verrouillés n'est lue.

**Questions et lectures fixées à l'avance** (`plan_09.json`) :
1. **Quel groupe de descripteurs porte l'excès de dispersion ?** Var(â) = Σ_g Cov(c_g, â), où c_g est la contribution linéaire du groupe g à â. On compare ces parts entre développement et verrouillés. Les groupes sont l'observation, le case-mix, les types d'unité, la structure et le modèle (moyenne et écart-type du score).
2. **Les hôpitaux verrouillés sont-ils hors du domaine du développement ?**
   * Distance de Mahalanobis régularisée D² sur les descripteurs standardisés par le développement, avec une corrélation rétrécie à moitié vers l'identité.
   * Seuil : 95e centile des distances *leave-one-out* des hôpitaux de développement.
   * Par descripteur : part des hôpitaux verrouillés hors de la plage [min ; max] du développement.
3. **Lecture « extrapolation compatible »** si les deux conditions suivantes sont vraies :
   * **E2** : au moins 25 % des hôpitaux verrouillés sont hors domaine. On en attendrait environ 5 % s'ils venaient du même domaine.
   * **E3** : parmi les hôpitaux verrouillés **dans** le domaine, Var(â) ne dépasse pas 1,5 fois celle du développement, ce qui situe l'excès chez les hôpitaux hors domaine.

In [ ]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION, INTÉGRITÉ, PLAN
# ============================================================
import sys, json, hashlib, importlib, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_DIR = Path("..")
DATA_DIR, SRC_DIR = PROJECT_DIR / "data_v1", PROJECT_DIR / "src"
RES02, RES06B, RES06C, RES06D, RES07 = (PROJECT_DIR / "results" / d for d in
    ("02_baselines", "06b_faisabilite_opcal", "06c_robustesse_opcal", "06d_amendement_puissance", "07_confirmation"))
RES_DIR = PROJECT_DIR / "results" / "09_domaine_extrapolation"
(RES_DIR / "figures").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR.resolve()))
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

protocol = json.loads((DATA_DIR / "splits" / "protocol.json").read_text(encoding="utf-8"))
assert protocol["hash"] == "d92099ea7c11d517"
FOLDS = sorted(int(k) for k in protocol["folds"])
DEV = sorted(int(h) for k in protocol["folds"] for h in protocol["folds"][k])
LOCKED = sorted(int(h) for h in protocol["locked_test_hospitals"])
SPEC_PATH, AMEND_PATH = RES06C / "specification_gelee.json", RES06D / "amendement_1.json"
OSF = (RES06D / "depot_osf.md").read_text(encoding="utf-8")
AMEND = json.loads(AMEND_PATH.read_text(encoding="utf-8"))
assert sha256(SPEC_PATH) in OSF and sha256(AMEND_PATH) in OSF, "Spécification ou amendement modifiés."
assert all(sha256(SRC_DIR / f) == h for f, h in AMEND["code_sha256"].items()), "Modules confirmatoires modifiés."
FROZEN = json.loads(SPEC_PATH.read_text(encoding="utf-8"))["meta_modele"]["descripteurs"]

PLAN = {"document": "Plan d'analyse descriptive du notebook 09 (domaine et extrapolation)",
        "statut": "exploratoire, descriptif, sans label verrouillé",
        "acces_verrouille": ["engagement_descripteurs_verrouilles.parquet", "engagement_predictions_patients_x_only.parquet (lp, sans label)",
                             "engagement_predictions_hopitaux_x_only.parquet (â_h)"],
        "groupes": ["observation", "case-mix", "types d'unité", "structure", "modèle"],
        "decomposition": "Var(â) = Σ_g Cov(c_g, â), c_g = Σ_{j∈g} β_j (z_j − m_j)/s_j, méta-modèle final du 07",
        "distance": "D² = z' R_s⁻¹ z, z standardisé par le développement, R_s = 0,5 R + 0,5 I ; seuil = P95 des distances leave-one-out du développement",
        "E2": "part des verrouillés hors domaine ≥ 0,25", "E3": "Var(â) des verrouillés dans le domaine ≤ 1,5 × Var(â) du développement",
        "lecture": "extrapolation compatible si E2 et E3 ; sinon non établie", "seuils": {"rho": 0.5, "q": 0.95, "E2": 0.25, "E3": 1.5}}
PLAN_PATH = RES_DIR / "plan_09.json"
if PLAN_PATH.exists():
    old = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
    assert {k: v for k, v in old.items() if k != "date"} == PLAN, "plan_09.json existe avec un autre contenu."
    PLAN = old
else:
    PLAN["date"] = datetime.now().isoformat(timespec="seconds")
    PLAN_PATH.write_text(json.dumps(PLAN, indent=2, ensure_ascii=False), encoding="utf-8")
PLAN_SHA, TH = sha256(PLAN_PATH), PLAN["seuils"]
print(f"Intégrité OK ; plan {PLAN_PATH.name} du {PLAN['date']} ({PLAN_SHA[:16]}…)")

## Étape 1 — Méta-modèle final (comme au 07) et contrôle de reproduction des â_h gelés

In [ ]:
# ============================================================
# ÉTAPE 1 — MÉTA-MODÈLE FINAL ET REPRODUCTION
# ============================================================
import opcal_feas as oc
importlib.reload(oc)
TR, _ = oc.load_transfer(RES02 / "preds", "x_only", FOLDS)
cal = oc.hospital_calibration(TR)
D_SITE = pd.read_csv(RES06B / "descripteurs_site.csv", index_col=0); D_SITE.index = D_SITE.index.astype(int)
D_DEV = D_SITE.join(cal[["mean_lp", "sd_lp"]]).loc[cal.index]

D_LOCK = pd.read_parquet(RES07 / "engagement_descripteurs_verrouilles.parquet").set_index("hospitalid")
P_LOCK = pd.read_parquet(RES07 / "engagement_predictions_patients_x_only.parquet")
assert "y" not in P_LOCK.columns, "Fichier de prédictions verrouillées inattendu (colonne y)."
g = P_LOCK.groupby("hospitalid")["lp"]
D_LOCK = D_LOCK.join(pd.DataFrame({"mean_lp": g.mean(), "sd_lp": g.std(ddof=0)}))
D_LOCK.index = D_LOCK.index.astype(int)
D_LOCK = D_LOCK.loc[LOCKED]
A_LOCK_GELE = pd.read_parquet(RES07 / "engagement_predictions_hopitaux_x_only.parquet").set_index("hospitalid")["a_hat"]

M = oc.SiteEB().fit(D_DEV[FROZEN].to_numpy(float), cal["citl"].to_numpy(), cal["var_citl"].to_numpy())
def contributions(D):
    Z = D[FROZEN].to_numpy(float)
    Z = np.where(np.isfinite(Z), Z, M.fill)
    return pd.DataFrame((Z - M.mz) / M.sz * M.beta, index=D.index, columns=FROZEN)
C_DEV, C_LOCK = contributions(D_DEV), contributions(D_LOCK)
A_DEV, A_LOCK = M.mu + C_DEV.sum(axis=1), M.mu + C_LOCK.sum(axis=1)
CONTROLES = []
oc.check("â_h verrouillés reproduits à l'identique (phase A du 07)",
         np.allclose(A_LOCK.to_numpy(), A_LOCK_GELE.reindex(A_LOCK.index).to_numpy(), rtol=1e-8, atol=1e-10), CONTROLES)
oc.check("Décomposition exacte : μ + Σ contributions = predict()", np.allclose(A_DEV, M.predict(D_DEV[FROZEN].to_numpy(float))[0]), CONTROLES)
assert all(r["ok"] for r in CONTROLES), "Reproduction en échec : ne pas interpréter."
print(f"Var(â) — développement (modèle final, en échantillon) {np.var(A_DEV):.4f} ; verrouillés {np.var(A_LOCK):.4f} ; "
      f"rapport {np.var(A_LOCK) / np.var(A_DEV):.2f}")

## Étape 2 — Quel groupe de descripteurs porte l'excès de dispersion ?

In [ ]:
# ============================================================
# ÉTAPE 2 — DÉCOMPOSITION DE Var(â) PAR GROUPE
# ============================================================
def group_of(c):
    if c in ("mean_lp", "sd_lp"):
        return "modèle"
    if c.startswith(("beds_cat", "teaching", "region_")):
        return "structure"
    if c.startswith("unit_"):
        return "types d'unité"
    if c.startswith(("age_mean", "female_share", "src_", "dx_")):
        return "case-mix"
    return "observation"
GROUP = pd.Series({c: group_of(c) for c in FROZEN})

def decompose(C, a):
    ac = a - a.mean()
    G = C.T.groupby(GROUP).sum().T
    return pd.DataFrame({"Var(c_g)": G.var(ddof=0), "Cov(c_g, â)": (G - G.mean()).mul(ac, axis=0).mean()})
dd, dl = decompose(C_DEV, A_DEV), decompose(C_LOCK, A_LOCK)
TAB_G = pd.concat({"développement": dd, "verrouillés": dl}, axis=1)
TAB_G[("excès", "Δ Cov(c_g, â)")] = dl["Cov(c_g, â)"] - dd["Cov(c_g, â)"]
TAB_G[("excès", "part de l'excès")] = TAB_G[("excès", "Δ Cov(c_g, â)")] / (np.var(A_LOCK) - np.var(A_DEV))
TAB_G = TAB_G.sort_values(("excès", "Δ Cov(c_g, â)"), ascending=False)
TAB_G.loc["total"] = [TAB_G[c].sum() for c in TAB_G.columns]
display(TAB_G.round(4))
GROUPE_PRINCIPAL = TAB_G.index[0]

cj_dev = (C_DEV - C_DEV.mean()).mul(A_DEV - A_DEV.mean(), axis=0).mean()
cj_lock = (C_LOCK - C_LOCK.mean()).mul(A_LOCK - A_LOCK.mean(), axis=0).mean()
TAB_J = pd.DataFrame({"groupe": GROUP, "Cov dév.": cj_dev, "Cov verrouillés": cj_lock, "Δ": cj_lock - cj_dev}).sort_values("Δ", ascending=False)
display(TAB_J.head(10).round(4))

## Étape 3 — Les hôpitaux verrouillés sont-ils hors du domaine du développement ?

In [ ]:
# ============================================================
# ÉTAPE 3 — DOMAINE : DISTANCES ET PLAGES
# ============================================================
def fill(D):
    Z = D[FROZEN].to_numpy(float)
    return np.where(np.isfinite(Z), Z, M.fill)
ZD, ZL = fill(D_DEV), fill(D_LOCK)
keep = ZD.std(0) > 1e-9
ZD, ZL, COLS = ZD[:, keep], ZL[:, keep], [c for c, k in zip(FROZEN, keep) if k]

def d2(Zref, Znew, rho=TH["rho"]):
    m, s = Zref.mean(0), Zref.std(0)
    s = np.where(s > 1e-9, s, 1.0)
    R = np.corrcoef(((Zref - m) / s).T)
    R = np.nan_to_num(R); np.fill_diagonal(R, 1.0)
    Ri = np.linalg.inv(rho * R + (1 - rho) * np.eye(len(m)))
    z = (Znew - m) / s
    return np.einsum("ij,jk,ik->i", z, Ri, z), z

D2_DEV = np.array([d2(np.delete(ZD, i, 0), ZD[i:i + 1])[0][0] for i in range(len(ZD))])     # leave-one-out
SEUIL = float(np.quantile(D2_DEV, TH["q"]))
D2_LOCK, Z_LOCK = d2(ZD, ZL)
HORS = D2_LOCK > SEUIL
part_hors = float(HORS.mean())

lo, hi = ZD.min(0), ZD.max(0)
out = (ZL < lo) | (ZL > hi)
TAB_D = pd.DataFrame({"groupe": GROUP[COLS].to_numpy(), "part hors plage": out.mean(0), "|z| moyen verrouillés": np.abs(Z_LOCK).mean(0)},
                     index=COLS).sort_values("part hors plage", ascending=False)
display(TAB_D.head(12).round(3))
TAB_GD = TAB_D.groupby("groupe")[["part hors plage", "|z| moyen verrouillés"]].mean().sort_values("part hors plage", ascending=False)
display(TAB_GD.round(3))

va_dev = float(np.var(A_DEV))
va_in, va_out = float(np.var(A_LOCK[~HORS])) if (~HORS).sum() > 1 else np.nan, float(np.var(A_LOCK[HORS])) if HORS.sum() > 1 else np.nan
rho_s = float(pd.Series(D2_LOCK).corr(pd.Series(np.abs(A_LOCK.to_numpy() - A_DEV.mean())), method="spearman"))
E2 = bool(part_hors >= TH["E2"])
E3 = bool(np.isfinite(va_in) and va_in <= TH["E3"] * va_dev)
LECTURE = "extrapolation compatible" if (E2 and E3) else "extrapolation non établie"
TAB_E = pd.DataFrame([
    {"lecture": "E2", "valeur": f"{part_hors:.0%} des verrouillés hors domaine (D² > {SEUIL:.1f}, P95 leave-one-out du dév. ; attendu ≈ 5 %)", "vrai": E2},
    {"lecture": "E3", "valeur": f"Var(â) verrouillés dans le domaine {va_in:.4f} ; hors domaine {va_out:.4f} ; développement {va_dev:.4f}", "vrai": E3},
]).set_index("lecture")
display(TAB_E)
print(f"Corrélation de Spearman entre D² et |â − moyenne| (verrouillés) : {rho_s:.2f}")
print(f"LECTURE : {LECTURE.upper()} ; groupe portant le plus l'excès de dispersion : {GROUPE_PRINCIPAL}")

## Étape 4 — Figure, contrôles et rapport

In [ ]:
# ============================================================
# ÉTAPE 4 — FIGURE, CONTRÔLES, RAPPORT
# ============================================================
oc.check("Aucun label ni cible de test verrouillé lu (fichiers de phase A uniquement, sans colonne y)", "y" not in P_LOCK.columns, CONTROLES)
oc.check("Plan inchangé pendant l'exécution", sha256(PLAN_PATH) == PLAN_SHA, CONTROLES)
CONTROLES_DF = pd.DataFrame(CONTROLES)
print(f"{int(CONTROLES_DF['ok'].sum())}/{len(CONTROLES_DF)} contrôles réussis")

import matplotlib.pyplot as plt
PAL = {"s1": "#2a78d6", "s2": "#eb6834", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781", "grid": "#e1e0d9"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": PAL["muted"],
                     "axes.labelcolor": PAL["ink2"], "xtick.color": PAL["ink2"], "ytick.color": PAL["ink2"], "font.size": 9})
gs = [g for g in TAB_G.index if g != "total"]
y = np.arange(len(gs))[::-1]
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.barh(y + 0.19, TAB_G.loc[gs, ("développement", "Cov(c_g, â)")], height=0.36, color=PAL["s1"], label="développement")
ax.barh(y - 0.19, TAB_G.loc[gs, ("verrouillés", "Cov(c_g, â)")], height=0.36, color=PAL["s2"], label="test verrouillé (sans label)")
ax.axvline(0, color=PAL["muted"], lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(gs)
ax.set_xlabel("part de Var(â) portée par le groupe : Cov(c_g, â)")
ax.grid(axis="x", color=PAL["grid"], lw=0.6); ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("Dispersion des prédictions OP-Cal par groupe de descripteurs", fontsize=9, color=PAL["ink"], loc="left")
plt.tight_layout(); plt.savefig(RES_DIR / "figures" / "dispersion_groupes.png", dpi=200); plt.show()

def md_table(df, digits=4):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(c) for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    esc = lambda x: str(x).replace("|", "\\|")
    head = "| " + " | ".join([esc(df.index.name or "")] + [esc(c) for c in df.columns]) + " |"
    return "\n".join([head, "|" + "---|" * (len(df.columns) + 1)] + ["| " + " | ".join([esc(i)] + [esc(v) for v in r]) + " |" for i, r in zip(df.index, df.to_numpy())])

rapport = f"""# Rapport 09 — domaine des hôpitaux verrouillés et dispersion des prédictions (descriptif, sans label)

Généré le {datetime.now():%Y-%m-%d %H:%M}. Protocole `{protocol['hash']}`. Plan `plan_09.json` du {PLAN['date']} (`{PLAN_SHA[:16]}…`).
**Exploratoire et descriptif.** Des hôpitaux verrouillés, seuls les fichiers gelés en phase A du 07 sont lus (descripteurs, lp sans label, â_h).

Var(â) : développement (modèle final, en échantillon) {va_dev:.4f} ; verrouillés {np.var(A_LOCK):.4f} (rapport {np.var(A_LOCK) / va_dev:.2f}).

## Quel groupe porte l'excès de dispersion ?

{md_table(TAB_G)}

Descripteurs dont la contribution augmente le plus :

{md_table(TAB_J.head(10))}

## Domaine du développement

{md_table(TAB_E)}

Corrélation de Spearman entre D² et |â − moyenne| chez les verrouillés : {rho_s:.2f}.

Parts hors plage [min ; max] du développement, par groupe :

{md_table(TAB_GD, 3)}

Descripteurs les plus souvent hors plage :

{md_table(TAB_D.head(12), 3)}

## Lecture : **{LECTURE}**

Groupe portant le plus l'excès de dispersion : **{GROUPE_PRINCIPAL}**. Il s'agit d'une description compatible ou non avec une
extrapolation du méta-modèle, pas d'une démonstration causale ; une dispersion plus forte peut aussi refléter une hétérogénéité réelle.

## Contrôles

{md_table(CONTROLES_DF.set_index('controle'))}
"""
(RES_DIR / "rapport_09.md").write_text(rapport, encoding="utf-8")
TAB_G.to_csv(RES_DIR / "decomposition_groupes.csv"); TAB_D.to_csv(RES_DIR / "domaine_descripteurs.csv")
print(f"Écrit : {RES_DIR / 'rapport_09.md'} | plan_09.json | decomposition_groupes.csv | domaine_descripteurs.csv | figures/dispersion_groupes.png")